# Categorical Interactions

本 Notebook 记录 Feature Engineering Round 2 中的 categorical × numerical gated features、categorical × categorical 显式组合，以及 CAT_NUM_GROUP 与 CAT2_GROUP 的完整 5-Fold 结果。Notebook 只读取和分析结果，不负责模型训练。

---

## Table of Contents

1. [实验背景与环境设置](#cat-setup)
   - 1.1 [实验目标](#cat-contract)
   - 1.2 [导入依赖与定位项目路径](#cat-imports)
2. [FE30–FE46 特征目录](#cat-catalog)
3. [交互特征实现原则](#cat-implementation)
4. [受控实验协议与运行方式](#cat-run)
5. [CAT_NUM_GROUP 结果](#cat-num-results)
6. [CAT2_GROUP 结果](#cat2-results)
7. [结果分析与 Reserve 规则](#cat-decisions)
8. [阶段小结与下一步](#cat-next)

---

<a id="cat-setup" name="cat-setup"></a>

## 1. 实验背景与环境设置

<a id="cat-contract" name="cat-contract"></a>

### 1.1 实验目标

CatBoost 已经接收原始 categorical features，但当前 effective `max_ctr_complexity` 为 1。Round 2 暂不改变模型配置，而是把特定交互作为独立 Feature Engineering hypothesis：

1. 某个 numerical feature 的风险曲线是否随 category 改变？
2. 明确的二阶 categorical combination 是否在 FE10 之上提供增量？

所有实验继续与相同 Fold 的 FE10 做配对比较。`max_ctr_complexity: 1 → 4` 另记为 MCC01，留到 Round 2 FE 收束后。

<a id="cat-imports" name="cat-imports"></a>

### 1.2 导入依赖与定位项目路径

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.round2_features import ROUND2_FEATURE_INFO

RESULTS_DIR = PROJECT_ROOT / "results" / "round2" / "first_stage"
SUMMARY_PATH = RESULTS_DIR / "summary.csv"

<a id="cat-catalog" name="cat-catalog"></a>

## 2. FE30–FE46 特征目录

第一阶段只运行 FE30、FE31、FE34、FE39、FE40 和 FE43–FE45。其余特征先作为 Conditional Reserve 保留。

In [ ]:
catalog_rows = []

for feature_id in [f"FE{i}" for i in range(30, 47)]:
    definition, status, first_stage_use = ROUND2_FEATURE_INFO[feature_id]
    catalog_rows.append(
        {
            "特征": feature_id,
            "定义": definition,
            "状态": status,
            "第一阶段用途": first_stage_use,
        }
    )

categorical_catalog = pd.DataFrame(catalog_rows)
display(categorical_catalog)

<a id="cat-implementation" name="cat-implementation"></a>

## 3. 交互特征实现原则

### Categorical × Numerical

不使用没有真实顺序含义的 `category_code × numerical`。例如 `gender × gaming` 会展开为：

```text
gaming_by_gender__Female
gaming_by_gender__Male
gaming_by_gender__Other
gaming_by_gender__Missing
```

类别匹配时保留原 numerical value；不匹配时为 NaN。原始 `gender` 和 `gaming_hours` 仍保留，同时增加 numerical missing flag。

### Categorical × Categorical

显式组合使用字符串，例如：

```text
Female__High
Male__Low
Missing__Medium
```

新组合列会明确加入 CatBoost `cat_features`。类别缺失统一填为 `"Missing"`。

<a id="cat-run" name="cat-run"></a>

## 4. 受控实验协议与运行方式

所有命令都从项目根目录执行。正式训练前先运行不训练模型的构造检查：

```bash
python -m src.round2_experiment --validate-only
```

然后按第一阶段优先级运行：

```bash
python -m src.round2_experiment CAT_NUM_GROUP
python -m src.round2_experiment CAT2_GROUP
```

两个 categorical group 完成后，回到 `05_fe10_extensions.ipynb`，最后运行低优先级的 O_GROUP。

不指定 `--fold` 时，每个实验自动运行完整 5-Fold。若只想运行一个 Fold：

```bash
python -m src.round2_experiment CAT_NUM_GROUP --fold 1
```

<a id="cat-num-results" name="cat-num-results"></a>

## 5. CAT_NUM_GROUP 结果

CAT_NUM_GROUP 同时加入 FE30、FE31、FE34、FE39 与 FE40。第一步只判断这一组人工 interaction 是否整体有增量。

In [ ]:
if SUMMARY_PATH.exists():
    round2_summary = pd.read_csv(SUMMARY_PATH)
    cat_num_summary = round2_summary.loc[
        round2_summary["experiment_id"] == "CAT_NUM_GROUP"
    ].copy()
    display(cat_num_summary)
else:
    round2_summary = pd.DataFrame()
    cat_num_summary = pd.DataFrame()
    print("当前还没有 Round 2 第一阶段汇总结果。")

In [ ]:
categorical_records = []

for path in sorted(RESULTS_DIR.glob("*_fold*.json")):
    result = json.loads(path.read_text(encoding="utf-8"))
    if result.get("experiment_id") in {"CAT_NUM_GROUP", "CAT2_GROUP"}:
        categorical_records.append(result)

categorical_folds = pd.DataFrame(categorical_records)

if categorical_folds.empty:
    print("当前还没有 categorical interaction 逐折结果。")
else:
    cat_num_folds = categorical_folds.loc[
        categorical_folds["experiment_id"] == "CAT_NUM_GROUP"
    ]
    display(
        cat_num_folds[[
            "fold",
            "reference_auc",
            "auc",
            "delta_vs_reference",
            "best_iteration",
        ]].sort_values("fold")
    )

<a id="cat2-results" name="cat2-results"></a>

## 6. CAT2_GROUP 结果

CAT2_GROUP 同时加入 FE43、FE44 与 FE45，验证显式二阶 categorical combination 是否有增量。

In [ ]:
if not round2_summary.empty:
    cat2_summary = round2_summary.loc[
        round2_summary["experiment_id"] == "CAT2_GROUP"
    ].copy()
    display(cat2_summary)
else:
    cat2_summary = pd.DataFrame()

if not categorical_folds.empty:
    categorical_pivot = categorical_folds.pivot(
        index="fold",
        columns="experiment_id",
        values="delta_vs_reference",
    )
    display(categorical_pivot)

    ax = categorical_pivot.plot(
        marker="o",
        figsize=(9, 5),
        color={
            "CAT_NUM_GROUP": "#2f6f9f",
            "CAT2_GROUP": "#d9a441",
        },
    )
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title("Categorical Interactions: Paired AUC Delta vs FE10")
    ax.set_xlabel("Fold")
    ax.set_ylabel("AUC delta vs FE10")
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

<a id="cat-decisions" name="cat-decisions"></a>

## 7. 结果分析与 Reserve 规则

### CAT_NUM_GROUP

如果完整 5-Fold 支持这一组，下一步先做 LOO，确认 gender、stress 和 impact 三个方向各自是否有贡献，再只唤醒对应 Reserve：

| 有贡献的方向 | Reserve 优先级 |
|---|---|
| Gender | FE32 → FE33 |
| Stress | FE38 / FE35 → FE36 → FE37 |
| Impact | FE42 → FE41 |

如果 CAT_NUM_GROUP 为负，所有 cat × num Reserve 保持冻结。

### CAT2_GROUP

如果 CAT2_GROUP 成立，先做 FE43–FE45 的 LOO，再考虑 FE46 三阶组合；如果为负，FE46 不运行。

<a id="cat-next" name="cat-next"></a>

## 8. 阶段小结与下一步

第一阶段只验证两个 family-level hypothesis，不在本 Notebook 中逐个运行 17 个 categorical interaction。是否进入 LOO 和 Conditional Reserve，必须同时参考完整 5-Fold、NULL noise、Fold 一致性与复杂度。